# Task 1.2 — Find a site where the drone and the field crew overlap

**Question:** which OFO drone mission has a stem-mapped field plot inside its footprint, so we can validate tree detections against ground truth?

**Plan:**
1. Search the OFO STAC collection for mission footprints (optionally narrowed to the Sierra Nevada to keep the result set small).
2. Load the ground-reference plot footprints.
3. Spatially join the two to find missions that actually contain a plot.
4. Score the overlaps against four criteria and pick one.

Fill in each `TODO(human)` below. Don't worry about getting it perfect on the first pass — this is meant to be iterated on.

In [ ]:
import pystac_client   # talks to the STAC API and runs the search
import geopandas as gpd  # everything downstream is a GeoDataFrame
from shapely.geometry import box  # for building a bounding box geometry

STAC_URL = "https://stac.cyverse.org"
COLLECTION = "Open Forest Observatory"  # confirm the exact collection id once you open the catalog

In [ ]:
def search_ofo_missions(bbox: tuple[float, float, float, float]) -> gpd.GeoDataFrame:
    """Search the OFO STAC collection and return mission footprints as a GeoDataFrame.

    Parameters
    ----------
    bbox : (minx, miny, maxx, maxy) in EPSG:4326 — e.g. a rough box around the Sierra Nevada.

    Returns
    -------
    GeoDataFrame, one row per STAC Item, with at least: item id, geometry (the mission
    footprint), and whatever properties you'll need later to judge each mission
    (e.g. a withheld_from_training flag, asset keys available).

    TODO(human):
    - Open the catalog with pystac_client.Client.open(STAC_URL)
    - Call .search(collections=[COLLECTION], bbox=bbox) and pull the items out
      (check the pystac-client docs for the right method — something returns
      an ItemCollection or an iterator of Items)
    - For each Item, grab item.id, item.geometry (or item.bbox), and item.properties
    - Build a GeoDataFrame from that (gpd.GeoDataFrame(..., geometry=..., crs="EPSG:4326"))
    - Print how many missions came back before you do anything else — sanity check
      the search actually hit something
    """
    raise NotImplementedError

In [ ]:
# TODO(human): pick a rough Sierra Nevada bbox (minx, miny, maxx, maxy in lon/lat) and run the search.
# sierra_bbox = (...)
# missions_gdf = search_ofo_missions(sierra_bbox)
# missions_gdf.head()

In [ ]:
def load_field_plots() -> gpd.GeoDataFrame:
    """Load OFO ground-reference plot footprints as a GeoDataFrame.

    TODO(human):
    - Browse https://openforestobservatory.org/data/ground-ref to find how plot
      footprints/locations are published (GeoPackage download? a listing you need
      to turn into points/polygons yourself?)
    - Load it with gpd.read_file(...)
    - Make sure the CRS is set and matches (or can be reprojected to match)
      missions_gdf before you join them
    """
    raise NotImplementedError

In [ ]:
# TODO(human): spatial join — which missions' footprints actually contain a field plot?
#
# Think about this before coding: do you want gpd.sjoin with predicate="contains"
# (mission contains plot) or "intersects"? What's the practical difference for a plot
# that sits right at the edge of a mission's flight footprint?
#
# plots_gdf = load_field_plots()
# overlaps = gpd.sjoin(missions_gdf, plots_gdf, predicate="...")
# overlaps

## Score the overlaps

For each mission/plot pair in `overlaps`, check against the four criteria from the plan:

1. Mission has both an orthomosaic **and** a canopy height model asset
2. Plot records species and live/dead status
3. You can see gray/red-brown dead crowns when you eyeball the orthomosaic
4. Ideally, the mission is flagged `withheld_from_training`

TODO(human): turn this into a small table or scoring column — doesn't need to be fancy,
even a markdown cell with your notes per candidate site is fine. Fall back to Emerald
Point (plot 0068) only if nothing else clears the bar — remember that's a less
independent v2 test since OFO used it to develop their own methods.